<a href="https://colab.research.google.com/github/Ajay-v44/AI-ML/blob/main/slm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


### What is `datasets` and why are we installing it?

This first cell installs a Python library called `datasets`. Think of it as a special toolbox that makes it super easy to download and work with many different kinds of data. In machine learning, we often need large amounts of data to train our models, and `datasets` helps us get that data quickly.

Here, the `pip install -U datasets` command ensures that the `datasets` library is installed or updated to its latest version in our Colab environment.

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

### Loading our dataset: TinyStories

After installing the `datasets` library, we use it to load a specific dataset. The line `from datasets import load_dataset` brings the function we need from the library.

Then, `ds = load_dataset("roneneldan/TinyStories")` downloads and loads the "TinyStories" dataset. This dataset contains very simple, short stories, which is perfect for teaching a basic language model how to generate text.

You might see a warning about `HF_TOKEN`. This is just a suggestion to set up a Hugging Face token for faster downloads and higher rate limits, but it's not strictly necessary for this notebook to run, as the dataset is public.

In [ ]:
!pip install tiktoken
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

enc = tiktoken.get_encoding("gpt2")

# Some functions from https://github.com/karpathy/nanoGPT/blob/master/data/openwebtext/prepare.py

def process(example):
    ids = enc.encode_ordinary(example['text']) # encode_ordinary ignores any special tokens
    out = {'ids': ids, 'len': len(ids)}
    return out

if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=8,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

tokenizing the splits (num_proc=8):   0%|          | 0/2119719 [00:00<?, ? examples/s]

tokenizing the splits (num_proc=8):   0%|          | 0/21990 [00:00<?, ? examples/s]

writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

### Preparing the data for our model (Tokenization)

Computers don't understand words directly. They need numbers. This section is all about converting our text stories into numerical representations that a machine learning model can process. This process is called **tokenization**.

1.  **`!pip install tiktoken`**: We install another library called `tiktoken`. This library is used for tokenizing text, often used with OpenAI's GPT models. It converts text into a sequence of numbers (tokens).

2.  **`import tiktoken`**: We import the `tiktoken` library.

3.  **`enc = tiktoken.get_encoding("gpt2")`**: We load a specific tokenizer model (the 'gpt2' encoding) which knows how to break down English text into tokens.

4.  **`process(example)` function**: This function takes a piece of text (an `example`) and:
    *   `ids = enc.encode_ordinary(example['text'])`: Converts the text into a list of numbers (IDs).
    *   `out = {'ids': ids, 'len': len(ids)}`: Stores these IDs and their length.

5.  **Creating `train.bin` (Data Storage)**:
    *   The `if not os.path.exists("train.bin")` block checks if we've already processed and saved our data. If not, it proceeds.
    *   `tokenized = ds.map(...)`: Applies our `process` function to every story in the `TinyStories` dataset. This converts all stories into lists of token IDs.
    *   The code then goes through each part (split like 'train' or 'validation') of the tokenized dataset. It takes all the token IDs and saves them efficiently into a binary file (like `train.bin` and `validation.bin`). These files store our entire dataset as long sequences of numbers, ready for the model to use.

In [ ]:
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y


### Getting Batches of Data for Training

When training a neural network, we don't feed the entire dataset at once. Instead, we train in small chunks called **batches**. This function, `get_batch(split)`, is responsible for preparing these batches.

1.  **`split`**: This argument tells the function whether to get data from the 'train' set or the 'validation' set (we need both for training and evaluating).
2.  **`np.memmap(...)`**: This part reads the numerical data from the `train.bin` or `validation.bin` files we created earlier. `memmap` is a memory-efficient way to access very large files without loading them entirely into RAM.
3.  **`ix = torch.randint(...)`**: This randomly selects starting positions (`ix`) within the data. Each starting position marks the beginning of a sequence our model will learn from.
4.  **`x = torch.stack(...)` and `y = torch.stack(...)`**: For each selected starting position, it extracts two sequences:
    *   `x`: This is the input sequence (e.g., "Once upon a ti").
    *   `y`: This is the target sequence (e.g., "nce upon a tim"). The model will try to predict `y` given `x`. Notice that `y` is just `x` shifted one token to the right. This is how language models learn to predict the next word.
5.  **`x.pin_memory().to(device, non_blocking=True)`**: This moves the data to the appropriate processing unit (either a GPU, if `device_type == 'cuda'`, or the CPU). GPUs are much faster for neural network calculations. `pin_memory` and `non_blocking` help make this transfer more efficient.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



### Building Our Language Model: The GPT Architecture

This is the core of our project – defining the neural network model itself. This code defines a simplified version of the **GPT (Generative Pre-trained Transformer)** architecture, similar to what powers models like ChatGPT, but much smaller.

Let's break down the main components:

*   **`LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`**: These are the fundamental building blocks of a Transformer model.
    *   **`LayerNorm`**: Helps stabilize training by normalizing the input to subsequent layers.
    *   **`CausalSelfAttention`**: This is the most crucial part. It allows the model to "pay attention" to different parts of the input sequence to understand context. "Causal" means it can only look at previous tokens, not future ones, which is essential for predicting the next token in a sequence.
    *   **`MLP` (Multi-Layer Perceptron)**: A standard feed-forward neural network that processes information within each block.
    *   **`Block`**: A combination of `LayerNorm`, `CausalSelfAttention`, and `MLP`. GPT models are made of many such blocks stacked on top of each other.

*   **`GPTConfig`**: This `dataclass` holds all the important settings (hyperparameters) for our GPT model, such as:
    *   `vocab_size`: How many unique words/tokens the model knows.
    *   `block_size`: The maximum length of the sequence the model can process at once (its "memory").
    *   `n_layer`: How many `Block`s are stacked.
    *   `n_head`: Number of "attention heads" in `CausalSelfAttention`, allowing the model to focus on different aspects of the input simultaneously.
    *   `n_embd`: The size of the internal representation (embedding) for each token.

*   **`GPT` class**: This is the main model. It puts all the pieces together:
    *   **`wte` (Word Token Embedding)**: Converts each token ID into a rich numerical vector (embedding). This is how the model understands the meaning of words.
    *   **`wpe` (Positional Embedding)**: Adds information about the position of each token in the sequence, as the attention mechanism itself doesn't inherently understand order.
    *   **`h` (Blocks)**: The stack of `Block`s that process the input and build up understanding.
    *   **`lm_head` (Language Model Head)**: This is the final layer that takes the processed information and predicts the probability of the next token in the vocabulary. It essentially translates the model's internal understanding back into a potential next word.
    *   **`forward(self, idx, targets=None)`**: This method defines how the model processes input (`idx`) during training or prediction. It takes a sequence of token IDs, passes them through the transformer blocks, and outputs `logits` (raw predictions) and `loss` (how wrong its predictions were).
    *   **`generate(self, idx, max_new_tokens, ...)`**: This method is for generating new text. Given a starting sequence (`idx`), it repeatedly predicts the next token, adds it to the sequence, and continues until it generates `max_new_tokens`. It uses `temperature` and `top_k` to control the randomness and quality of the generated text.

In [ ]:
config = GPTConfig(
    vocab_size=50257,     # use the tokenizer's vocab size
    block_size=128,       # or whatever context size you're training with
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.1,
    bias=True
)

model = GPT(config)

### Initializing the GPT Model

After defining the structure of our `GPT` model, this cell sets up its specific configuration and creates an instance of the model.

1.  **`config = GPTConfig(...)`**: We create an object called `config` using our `GPTConfig` blueprint. Here, we specify the values for our model's hyperparameters:
    *   `vocab_size=50257`: The number of unique tokens the model can recognize (this is standard for the 'gpt2' tokenizer).
    *   `block_size=128`: The maximum number of tokens the model can look at simultaneously to understand context.
    *   `n_layer=6`: The model will have 6 Transformer `Block`s stacked.
    *   `n_head=6`: Each attention mechanism will have 6 "heads" for parallel processing.
    *   `n_embd=384`: The internal dimension for representing tokens.
    *   `dropout=0.1`: A technique to prevent overfitting (making the model too good at predicting training data but bad at new data) by randomly ignoring some connections during training.
    *   `bias=True`: Whether to use bias terms in linear layers.

2.  **`model = GPT(config)`**: We create an actual `GPT` model, named `model`, using the `config` we just defined. At this point, the model's internal parameters (weights) are randomly initialized, and it's ready to be trained.

In [ ]:
def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

### Estimating Loss: How Well is Our Model Doing?

During training, we need a way to check how well our model is learning. This function, `estimate_loss(model)`, does exactly that. It calculates the average 'loss' (how wrong the model's predictions are) on both the training data and a separate validation data set.

1.  **`model.eval()`**: Before evaluating, we put the model into "evaluation mode." This disables certain behaviors that are only active during training, like `dropout` (which randomly ignores connections) and `BatchNorm` (if present).
2.  **`torch.inference_mode()`**: This tells PyTorch that we're just calculating, not training, so it doesn't need to store extra information for backpropagation (updating model weights). This saves memory and speeds up evaluation.
3.  **Looping through `split`s**: It calculates loss for both `train` and `val` (validation) splits.
4.  **`get_batch(split)`**: We use our `get_batch` function to get a small chunk of data for evaluation.
5.  **`logits, loss = model(X, Y)`**: We feed the input (`X`) to the model and get its predictions (`logits`) and the calculated `loss` (how far off the predictions were from the actual `Y`).
6.  **`losses[k] = loss.item()`**: We store the loss for this batch.
7.  **`out[split] = losses.mean()`**: After evaluating several batches, we take the average loss for that split.
8.  **`model.train()`**: Finally, we switch the model back to "training mode" so it's ready for the next training steps.

In [ ]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 1e-4 #more stable training, earlier 1e-4
max_iters = 20000 #increase from 25000
warmup_steps = 1000 #smoother initial train, earlier 100
min_lr = 5e-4 #lower rate, earlier 5e-4
eval_iters = 500 # increased from 100
batch_size = 32 # changed from 16, better gradient estimate
block_size = 128 #changed from 64, capture longer range dependencies

gradient_accumulation_steps = 32 # reduced from 50

device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
# note: float16 data type will automatically use a GradScaler

# How to use autocast https://wandb.ai/wandb_fc/tips/reports/How-To-Use-Autocast-in-PyTorch--VmlldzoyMTk4NTky
#dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

### Training Configuration: Setting Up for Learning

This section defines various settings (hyperparameters) that control how our model will be trained. Think of these as the 'recipe' for teaching the model.

*   **`learning_rate`**: How big a step the model takes to adjust its internal parameters during training. A smaller learning rate means slower, more careful adjustments; a larger one means faster, potentially unstable adjustments.
*   **`max_iters`**: The total number of training steps (iterations) the model will go through.
*   **`warmup_steps`**: For the initial steps, the learning rate will gradually increase to prevent early training instability.
*   **`min_lr`**: The minimum learning rate the model can drop to during training.
*   **`eval_iters`**: How often we pause training to evaluate the model's performance on both training and validation data.
*   **`batch_size`**: The number of input-output pairs the model processes at once before updating its parameters.
*   **`block_size`**: This is the context window, the maximum sequence length the model can consider when making a prediction. (Already defined in `GPTConfig`).
*   **`gradient_accumulation_steps`**: To effectively use larger batch sizes without needing more GPU memory, we can accumulate gradients over several smaller batches before performing a single parameter update. This helps simulate a larger `batch_size`.
*   **`device`, `device_type`**: This checks if a GPU (CUDA) is available. If so, it uses the GPU for faster computations; otherwise, it defaults to the CPU.
*   **`dtype`, `ptdtype`**: Specifies the data type for calculations (e.g., `float16` for faster computations with less memory on compatible GPUs). `bfloat16` is a specialized floating-point format that balances precision and memory usage, often used in machine learning.
*   **`ctx = nullcontext() if ... else torch.amp.autocast(...)`**: This sets up automatic mixed precision training. If a GPU is available and a `float16` or `bfloat16` data type is chosen, PyTorch will automatically use lower precision for some operations to speed up training and reduce memory usage, while maintaining numerical stability.
*   **`torch.set_default_device(device)`**: Sets the default device for new tensors.
*   **`torch.manual_seed(42)`**: Ensures that random operations (like initial model weights) are reproducible. If you run the code again with the same seed, you'll get the same random numbers.

In [ ]:
from torch.optim.lr_scheduler import LinearLR,SequentialLR, CosineAnnealingLR

##PUT IN WEIGHT DECAY, CHANGED BETA2 to 0.95
optimizer =  torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

scheduler_warmup = LinearLR(optimizer, total_iters = warmup_steps) #Implement linear warmup
scheduler_decay = CosineAnnealingLR(optimizer,T_max = max_iters - warmup_steps, eta_min = min_lr) #Implement lr decay
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay

# https://stackoverflow.com/questions/72534859/is-gradscaler-necessary-with-mixed-precision-training-with-pytorch
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

/tmp/ipykernel_6772/2132813893.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


### Setting Up the Optimizer and Learning Rate Scheduler

This section deals with how our model's parameters are updated during training and how its learning rate changes over time.

*   **`optimizer = torch.optim.AdamW(...)`**: This creates an **optimizer**. The optimizer's job is to adjust the model's internal weights based on the calculated `loss` so that the model gets better at its task. `AdamW` is a popular and effective optimizer.
    *   `lr=learning_rate`: The initial learning rate.
    *   `betas=(0.9, 0.95)`: Parameters for the Adam optimizer.
    *   `weight_decay=0.1`: A regularization technique that helps prevent overfitting by pushing model weights towards smaller values.
    *   `eps=1e-9`: A small number to prevent division by zero.

*   **`LinearLR`, `CosineAnnealingLR`, `SequentialLR`**: These are **learning rate schedulers**. They dynamically change the learning rate during training, which can help the model converge faster and achieve better performance.
    *   **`scheduler_warmup = LinearLR(...)`**: For the first `warmup_steps`, the learning rate will linearly increase from a very small value up to the initial `learning_rate`.
    *   **`scheduler_decay = CosineAnnealingLR(...)`**: After the warmup, the learning rate will gradually decrease following a cosine curve down to `min_lr` over the remaining training steps.
    *   **`scheduler = SequentialLR(...)`**: This combines the `warmup` and `decay` schedulers, ensuring a smooth transition between them.

*   **`scaler = torch.cuda.amp.GradScaler(...)`**: This is specifically for **mixed precision training** when using `float16`. It helps prevent numerical underflow (numbers becoming too small to represent accurately) during gradient calculations, which can happen with lower precision floating-point numbers.

In [ ]:
best_val_loss = float('inf')
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []

model = model.to(device)

for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        # Step scheduler AFTER optimizer step
        scheduler.step()

  0%|          | 0/20000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 500: train loss 9.4197, val loss 9.4251
The current learning rate: 0.00007
Epoch 1000: train loss 8.4603, val loss 8.4635
The current learning rate: 0.00010
Epoch 1500: train loss 7.5155, val loss 7.5132
The current learning rate: 0.00010
Epoch 2000: train loss 6.6729, val loss 6.6679
The current learning rate: 0.00010
Epoch 2500: train loss 5.9845, val loss 5.9775
The current learning rate: 0.00011


### The Training Loop: Teaching the Model to Write Stories

This is where the magic happens – the model learns from the data! This `for` loop represents the entire training process, iterating `max_iters` times (in our case, 20,000 epochs).

1.  **`best_val_loss = float('inf')`, `best_model_params_path`**: We initialize `best_val_loss` to a very high number and define a file path to save the model parameters that result in the lowest validation loss. This ensures we keep the best version of our model.

2.  **`model = model.to(device)`**: Moves the model itself to the chosen device (GPU or CPU).

3.  **`for epoch in tqdm(range(max_iters))`**: This is the main training loop. `tqdm` is a library that provides a nice progress bar.

4.  **Evaluation (every `eval_iters` epochs)**:
    *   `if epoch % eval_iters == 0 and epoch != 0:`: Every 500 epochs (after the first one), we pause training to evaluate the model.
    *   `losses = estimate_loss(model)`: Calls our `estimate_loss` function to get the current training and validation loss.
    *   It then prints these losses and the current learning rate.
    *   **Saving the best model**: `if losses['val'] < best_val_loss: ...` If the current validation loss is better than any previous loss, we update `best_val_loss` and save the model's current state (its learned weights) to `best_model_params_path`.

5.  **Getting a batch of data**: `X, y = get_batch("train")` retrieves input (`X`) and target (`y`) sequences from the training data.
    *   `X, y = X.to(device), y.to(device)`: Moves the data to the appropriate device.

6.  **Forward Pass and Loss Calculation**: `with ctx: logits, loss = model(X, y)`
    *   The input `X` is fed to the `model` to get predictions (`logits`) and the `loss`.
    *   `loss = loss / gradient_accumulation_steps`: The loss is scaled down because we are accumulating gradients. This ensures that the average gradient over the `gradient_accumulation_steps` is correct.
    *   `scaler.scale(loss).backward()`: This calculates the gradients (how much each parameter needs to change) for the current batch. `scaler.scale()` is part of mixed precision training.

7.  **Optimizer Step (Parameter Update)**:
    *   `if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):` This condition means we only update the model's parameters after accumulating gradients for `gradient_accumulation_steps` batches, or at the very end of training.
    *   `torch.nn.utils.clip_grad_norm_`: Prevents gradients from becoming too large, which can make training unstable.
    *   `scaler.step(optimizer)`: Updates the model's parameters based on the accumulated gradients. Again, `scaler.step()` is for mixed precision.
    *   `scaler.update()`: Updates the `GradScaler` for the next iteration.
    *   `optimizer.zero_grad(set_to_none=True)`: Clears the gradients from the previous step to prepare for the next batch.

8.  **Learning Rate Scheduler Step**: `scheduler.step()` updates the learning rate based on the predefined schedule (warmup then decay).

In [ ]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Visualizing Training Progress

After all that training, it's very helpful to see how our model improved over time. This cell uses `matplotlib` to plot the training and validation losses.

1.  **`import matplotlib.pyplot as plt`**: Imports the plotting library.
2.  **`train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]`** and **`validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]`**: During training, the losses were stored as PyTorch tensors, potentially on the GPU. We need to move them to the CPU (`.cpu()`) and detach them from the computation graph (`.detach()`) before converting them to standard Python numbers for plotting.
3.  **`plt.plot(...)`**: These lines draw two lines on the plot:
    *   A green line for the training loss.
    *   A red line for the validation loss.
4.  **`plt.xlabel(...)`, `plt.ylabel(...)`, `plt.legend()`, `plt.show()`**: These add labels to the axes, a legend to explain which line is which, and finally display the plot.

In [ ]:
#Load the model
model = GPT(config)  # re-create the model with same config
device =  "cuda" if torch.cuda.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device))) # load best model states


### Loading the Best Model

During training, we saved the model's parameters whenever it achieved the lowest validation loss. This cell loads those best-performing parameters back into a new model instance.

1.  **`model = GPT(config)`**: We first create a fresh `GPT` model with the same `config` as before. This model has randomly initialized weights.
2.  **`device = "cuda" if ... else "cpu"`**: Determines the device to load the model onto.
3.  **`model.load_state_dict(torch.load(best_model_params_path, ...))`**: This is the crucial step. It loads the saved parameters from `best_model_params_path` into our new `model` instance. Now, this `model` has the knowledge it gained during its best training performance.

In [ ]:
sentence = "Once upon a time there was a pumpkin."
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))

### Generating a Story (Example 1)

Now that our model is trained and loaded, we can ask it to generate new text! This cell provides a starting phrase and lets the model continue the story.

1.  **`sentence = "Once upon a time there was a pumpkin."`**: This is our initial prompt or "seed" for the story.
2.  **`context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))`**: We take our `sentence`, tokenize it into numbers using `enc.encode_ordinary()`, convert it into a PyTorch tensor, and add an extra dimension (`unsqueeze(dim=0)`) because the model expects batches of input.
3.  **`y = model.generate(context, 200)`**: We call the `generate` method of our `model`. It takes our `context` (the starting sentence) and `200` (the number of new tokens we want the model to generate).
4.  **`print(enc.decode(y.squeeze().tolist()))`**: The `generate` method returns a tensor of token IDs. We convert these back to a list of numbers, then use `enc.decode()` to turn those numbers back into human-readable text, and finally print the generated story.

In [ ]:
sentence = "A little girl went to the woods"
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))

### Generating Another Story (Example 2)

This cell is identical to the previous one, but with a different starting sentence, allowing us to see how the model generates text given a new prompt.

It demonstrates the model's ability to create diverse continuations based on different initial contexts.

In [ ]:
from google.colab import runtime
runtime.unassign()

### Unassigning Colab Runtime

**`from google.colab import runtime`** and **`runtime.unassign()`**:

This is a Colab-specific command. `runtime.unassign()` disconnects your Colab notebook from its allocated resources (like GPU and RAM). This is useful if you're done with a computationally intensive task and want to free up resources or avoid exceeding your usage limits. It essentially restarts the backend environment for the notebook, clearing all variables and states.